# VectorDrugMatch - final Colab run (PrimeKG, both benchmark variants)

Run the cells **top to bottom**. Everything is driven by `scripts/run_full_pipeline.py`; the code is **pinned to commit `1c85d98a35`** so the run is reproducible.
Before running: that commit must be pushed to GitHub (`origin/mukul`), and your private files must be in Drive (Step 3).
Training happens only here; local work produced the data layer, audits and exporters. Read `Docs/DATA_AND_PIPELINE_READINESS.md` first.
> **Record of the 2026-11-11 run.** This saved copy contains the outputs of the Colab session. The session started from commit `b86df0d` and code fixes were then applied in the runtime (stabilised GCN, logits-space BCE, FP32 islands for RGCN/HGT under mixed precision, relation-level GNN explanations); they were committed afterwards as `1c85d98`, the **code of record** now pinned above. Step 2's library cell was replaced during the session (PyG with `pyg-lib`, no `torch_cluster` build). See `Docs/PROJECT_STATUS.md`.


## Step 1 - GPU
Runtime > Change runtime type > T4 GPU. Expect `GPU available: True`.

In [1]:
import torch
print('torch', torch.__version__, '| GPU available:', torch.cuda.is_available())
!nvidia-smi | head -15

torch 2.14.1+cu130 | GPU available: True
Tue Nov 10 22:49:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8             13W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------

## Step 2 - Drive, code (pinned commit) and libraries

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/VDM_Data /content/drive/MyDrive/VDM_Outputs
%cd /content
!rm -rf VectorDrugMatch
!git clone https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd VectorDrugMatch
PIN = '1c85d98a356d62d232aa8f8e3acf5ac497c2532a'
!git checkout $PIN
!git log -1 --oneline

Mounted at /content/drive
/content
Cloning into 'VectorDrugMatch'...
remote: Enumerating objects: 690, done.
remote: Counting objects: 100% (690/690), done.
remote: Compressing objects: 100% (329/329), done.
remote: Total 690 (delta 378), reused 650 (delta 340), pack-reused 0 (from 0)
Receiving objects: 100% (690/690), 1.02 MiB | 14.47 MiB/s, done.
Resolving deltas: 100% (378/378), done.
/content/VectorDrugMatch
Note: switching to 'b86df0db7ce0985f798f417b71e6c48c03773046'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

HEAD

In [9]:

# Install the main project dependencies
%pip install -q rdkit mlflow optuna shap xgboost lightgbm loguru pyyaml torch-geometric tabulate scikit-learn lxml matplotlib pymupdf pyarrow

# Verify the PyTorch Geometric installation
import torch
import torch_geometric
import pyg_lib

print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("pyg-lib:", pyg_lib.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

assert torch.cuda.is_available()
print("STEP 2 DEPENDENCIES VERIFIED")


PyTorch: 2.14.1+cu130
PyG: 2.8.1
pyg-lib: 0.9.0+pt214cu130
CUDA available: True
GPU: Tesla T4
STEP 2 DEPENDENCIES VERIFIED


In [8]:
!mkdir -p outputs && pip freeze > outputs/pip_freeze_colab.txt && wc -l outputs/pip_freeze_colab.txt

748 outputs/pip_freeze_colab.txt


## Step 3a - Check the Drive upload BEFORE anything expensive (PASS/FAIL)
Checks every expected file in `MyDrive/VDM_Data/` (existence, size, SHA-256). **Stop on `RESULT: FAIL`**: fix the upload, then run this cell again. `EXTRA` lines name files the notebook does not read; a restricted file (for example `primekg_kg.csv`) makes the result FAIL: delete it from Drive.
Hashing reads about 340 MB from Drive; set `HASH = False` to check names and sizes only.

In [10]:
import hashlib, json, os
HASH = True
ROOT = '/content/drive/MyDrive/VDM_Data'
EXPECTED = json.loads('[["mondo.obo",53442278,"b5a3d848578694601dffb1c2866798d4f6f4b4a12c315030c4d4c62ffe4e7e22","R"],["drugbank_smiles_cache.json",546885,"31826e050a9b8f31584a194eca2dffbe78873adf0b4fa22dd95e9ae13211b3bc","R"],["drugbank_cid_map.json",153352,"1d036a3f15632e52293faf2fa1d0747be9f3bfa7b48370c35fec793c1a3b4a16","R"],["B09474-eng.pdf",1364687,"69daf7c8399b85978ba85e143d3af34e6ccf7f3d493b1e056daa7ef909d64eb2","R"],["downloads/hai_price_surveys/delhi_2011-07/affordability_excel.xls",31388,"fa38ac42d2c9e92d4e0a93ffe8474f9540fe4538a60705379b2ea7cd0c146c97","R"],["downloads/hai_price_surveys/delhi_2011-07/median_price_by_medicine_excel.xls",35436,"30ac3f147bbde4781ffaf371737a78283ba6a3692b1291ea5554b412324a222c","R"],["downloads/hai_price_surveys/delhi_2011-07/price_availability_survey_detail_excel (1).php",14861,"decbe86e5717914e38d19188bb2ef2a05745df6b578859e27b6a6ba739ae7d0e","R"],["downloads/hai_price_surveys/delhi_2011-07/price_availability_survey_detail_excel (2).php",15095,"3568934f3fcb5e91da60f3053cbcebf8344f9376b20189af92aceed351b027a5","R"],["downloads/hai_price_surveys/delhi_2011-07/price_availability_survey_detail_excel.php",14558,"1a8155f75ece38246a9b8ac50e29dd6516cfe0580997b55082b95cf0b2d964f0","R"],["downloads/hai_price_surveys/karnataka_2004-11/affordability_excel (2).xls",28826,"a35979eed83ac7b38b59e6b33f7fe47a4377a948f1cf2662ee459dfeb8b18755","R"],["downloads/hai_price_surveys/karnataka_2004-11/median_price_by_medicine_excel (2).xls",31679,"f264dbe13ea6cdbcd885360f204d01f051a25e69cf58776dd1a2b5093d13f51a","R"],["downloads/hai_price_surveys/karnataka_2004-11/price_availability_survey_detail_excel (6).php",16753,"f02e923721b2ccbe94cbbc1bb8b0d4f6488ae9cfb21c9c4d1e78316c71369fa0","R"],["downloads/hai_price_surveys/karnataka_2004-11/price_availability_survey_detail_excel (7).php",17159,"68562b18a173698fb74de7a35ba3cbbe49ef30546742eb5f41402511c1debdc4","R"],["downloads/hai_price_surveys/karnataka_2004-11/price_availability_survey_detail_excel (8).php",18013,"df47283e8a851b9e4244ff429e99dd8838abde6f27e206467fab01cf22b2d690","R"],["downloads/hai_price_surveys/maharashtra_2005-01/affordability_excel (1).xls",30506,"131a88c01d08aaeb4f9d034faf590311dfe1f4bf5878ac1859549371edab5ce7","R"],["downloads/hai_price_surveys/maharashtra_2005-01/median_price_by_medicine_excel (1).xls",30137,"1097885b7332fd5b451cc530c2a341f5421e025fab33b0939bc4868de4f42a8e","R"],["downloads/hai_price_surveys/maharashtra_2005-01/price_availability_survey_detail_excel (3).php",15962,"37085b78444f087750b435735d74af58065032e27b27e9687765b79d253bd722","R"],["downloads/hai_price_surveys/maharashtra_2005-01/price_availability_survey_detail_excel (4).php",16365,"d36b8906cd74e70ce0629a23440fb69e144d6128c13c47112e2f5657ed537bdb","R"],["downloads/hai_price_surveys/maharashtra_2005-01/price_availability_survey_detail_excel (5).php",17180,"bc2304a195b8247d9ed3579aa1e8e26062d01e4c426a4ae73316d47af617e515","R"],["downloads/ihme_gbd/IHME-GBD_2023_DATA-9feb184e-1.zip",870,"fcbabec0eee2a201f75279506275d62b132353ba96ad321f8de544b81e73edce","R"],["downloads/ihme_gbd/ihme_ntd_dalys_2023.csv",3155,"1922a4cdf30d585a1d50bddb7144fe837011de2e4b3a639b50fd93fcb8d954f5","R"],["downloads/ihme_gbd/ihme_ntd_dalys_2023_citation.txt",327,"339dc2be55e1576d87c0a9640ee20865e222e4f7d19e6d30a7731a0f8e476874","R"],["downloads/open_targets_release_26.09/clinical_indication.parquet",4989955,"e10ced7ce09495cd3fa6e7cb94e81e083aed78ee399424730d567243b49a38d8","R"],["downloads/open_targets_release_26.09/clinical_report.parquet",75924085,"ba0ea48b129182de119f5d676bb1483e6225ba65857a31b7319b65c0e6b8fa25","R"],["downloads/open_targets_release_26.09/drug_mechanism_of_action.parquet",322830,"8484accd3383837e0eaac2164eed89991daf58dc93f0a599072fb7e5ef4f6c28","R"],["downloads/open_targets_release_26.09/drug_molecule.parquet",6867467,"be65749aa1dbcc68d0930f8ac18126cec5c2bc2c0f232f11e8530dad4ba0585c","R"],["frozen_processed/primekg/disease_features.npy",205088,"c7a222863c6d833faded670de6718bd2b395768c29ac2eb3ec4fae0603f61032","R"],["frozen_processed/primekg/diseases.csv",2072360,"4d29b7baa3ce238d1c7e384680e93dfcf12ebc9d4bf4faeef16cea65f1fdc464","R"],["frozen_processed/primekg/drug_features.npy",65183872,"b03ccc4eaf5a83458fb9e1165a79b24d093b719a6e1b6c237911063b322d69ca","R"],["frozen_processed/primekg/drugs.csv",778957,"6fdd280a32eb3fc377907a316ae5fe325aa6c2cdb77e4e83ea4a4c0970d460ab","R"],["frozen_processed/primekg/edges.csv",29038960,"19cb372c2cab1bcf9e0ee472d32c613609476f6a69337096af0b72a405b53442","R"],["frozen_processed/primekg/genes.csv",350597,"ad20206067029a5b96046973defcdb38d91e1341eb2bc35550a0e138a8714ef0","R"],["frozen_processed/primekg/graph_metadata.json",258,"fbf323fa476555c21ee016b61f1ade1d4dc68260c42e1cf0c184dc9bcc2fc5c3","R"],["frozen_processed/primekg/primekg_provenance.json",2745,"66d26e9e25fcb1a9ac670fe7c675ed2cf71dd2311864a12cfb29e425d26f8556","R"],["frozen_processed/primekg/splits/seed42.manifest.json",3176,"9493dc5bdb515eea8f9e9ea6c6251e910eafd616433ef92d4fd84a6487eb72b2","R"],["frozen_processed/primekg/splits/seed42.npz",313153,"1475cdf93b24a91ff67b7c822e821a2688bdb3605d334fe06977e19b9299934b","R"],["frozen_processed/primekg_structcov/disease_features.npy",205088,"c7a222863c6d833faded670de6718bd2b395768c29ac2eb3ec4fae0603f61032","R"],["frozen_processed/primekg_structcov/diseases.csv",2072360,"4d29b7baa3ce238d1c7e384680e93dfcf12ebc9d4bf4faeef16cea65f1fdc464","R"],["frozen_processed/primekg_structcov/drug_features.npy",65183872,"b03ccc4eaf5a83458fb9e1165a79b24d093b719a6e1b6c237911063b322d69ca","R"],["frozen_processed/primekg_structcov/drugs.csv",778957,"6fdd280a32eb3fc377907a316ae5fe325aa6c2cdb77e4e83ea4a4c0970d460ab","R"],["frozen_processed/primekg_structcov/edges.csv",29038960,"19cb372c2cab1bcf9e0ee472d32c613609476f6a69337096af0b72a405b53442","R"],["frozen_processed/primekg_structcov/genes.csv",350597,"ad20206067029a5b96046973defcdb38d91e1341eb2bc35550a0e138a8714ef0","R"],["frozen_processed/primekg_structcov/graph_metadata.json",258,"fbf323fa476555c21ee016b61f1ade1d4dc68260c42e1cf0c184dc9bcc2fc5c3","R"],["frozen_processed/primekg_structcov/primekg_provenance.json",2745,"66d26e9e25fcb1a9ac670fe7c675ed2cf71dd2311864a12cfb29e425d26f8556","R"],["frozen_processed/primekg_structcov/splits/seed42.manifest.json",1078,"049792c4e0b0e5c3471ce18e4a330a5e69ee1666749424d72f38c52c474165de","R"],["frozen_processed/primekg_structcov/splits/seed42.npz",1417940,"1704266b94029fc085218b9733225f6507a887e6a46a8afafe5885de12451842","R"]]')
FORBIDDEN = ('primekg_kg.csv', 'drugbank_vocabulary.csv', 'full database.xml', 'full_database.xml', '.env')
bad_req, bad_rec, ok = [], [], 0
for path, size, sha, lvl in EXPECTED:
    f = os.path.join(ROOT, path); why = None
    if not os.path.isfile(f): why = 'MISSING'
    elif os.path.getsize(f) != size: why = f'SIZE {os.path.getsize(f)} != {size}'
    elif HASH and hashlib.sha256(open(f, 'rb').read()).hexdigest() != sha: why = 'HASH DIFFERS'
    if why is None: ok += 1
    else:
        print(f"FAIL [{'REQUIRED' if lvl == 'R' else 'RECOMMENDED'}] {path}: {why}"); (bad_req if lvl == 'R' else bad_rec).append(path)
known = {p for p, *_ in EXPECTED}
extra = []
for d, _, fs in os.walk(ROOT):
    for fn in fs:
        rel = os.path.relpath(os.path.join(d, fn), ROOT).replace(os.sep, '/')
        if rel not in known:
            extra.append(rel); print('EXTRA', rel, '-> RESTRICTED/NOT NEEDED: remove' if fn in FORBIDDEN else '-> not read by the notebook')
print(f'{ok}/{len(EXPECTED)} OK | REQUIRED problems: {len(bad_req)} | RECOMMENDED problems: {len(bad_rec)} | extra: {len(extra)} | hashes checked: {HASH}')
bad_extra = [p for p in extra if os.path.basename(p) in FORBIDDEN]
print('RESULT:', 'PASS' if not bad_req and not bad_extra else 'FAIL')
assert not bad_req and not bad_extra, 'fix the Drive upload (see FAIL/EXTRA lines above) before continuing'


46/46 OK | REQUIRED problems: 0 | RECOMMENDED problems: 0 | extra: 0 | hashes checked: True
RESULT: PASS


## Step 3 - Put your private/manual inputs in place (from Drive)
Everything comes from `MyDrive/VDM_Data/` (exact list: `Docs/GOOGLE_DRIVE_UPLOAD_GUIDE.md`; Step 3a checks it):
`mondo.obo` (the frozen MONDO release; **required**, otherwise the latest MONDO is downloaded and the benchmark can drift), `drugbank_smiles_cache.json`, `drugbank_cid_map.json`,
`downloads/` (Open Targets, HAI, IHME), `B09474-eng.pdf` and optionally `who_eml.csv`, and `frozen_processed/` (safety copy of the frozen benchmark tables).
Missing optional items are fine: the matching equity components are reported as **missing**, never as zero.

In [11]:
import os, shutil
D = '/content/drive/MyDrive/VDM_Data'
os.makedirs('data/external', exist_ok=True)
if os.path.isdir(f'{D}/downloads'):
    shutil.copytree(f'{D}/downloads', 'data/external/downloads', dirs_exist_ok=True); print('copied downloads/')
else:
    print('no downloads/ in Drive: Open Targets, HAI and IHME steps will be skipped (optional)')
# Open Targets: Drive holds ONE copy (open_targets_release_26.09/); recreate the three files the indication loader reads (identical bytes)
rel, loc = 'data/external/downloads/open_targets_release_26.09', 'data/external/downloads/open_targets'
if os.path.isdir(rel) and not os.path.isdir(loc):
    os.makedirs(loc)
    for a, b in (('clinical_indication.parquet', '00000000.parquet'), ('drug_molecule.parquet', '00000000 (1).parquet'), ('drug_mechanism_of_action.parquet', '00000000 (2).parquet')):
        shutil.copy(f'{rel}/{a}', f'{loc}/{b}')
    print('recreated downloads/open_targets/ from open_targets_release_26.09/')
for fn in ('mondo.obo', 'who_eml.csv', 'B09474-eng.pdf', 'drugbank_smiles_cache.json', 'drugbank_cid_map.json'):
    if os.path.exists(f'{D}/{fn}'):
        shutil.copy(f'{D}/{fn}', f'data/external/{fn}'); print('copied', fn)
    else:
        print('NOT in Drive:', fn)
assert os.path.exists('data/external/mondo.obo'), 'upload mondo.obo (frozen MONDO release 2026-11-03) to MyDrive/VDM_Data first'

copied downloads/
recreated downloads/open_targets/ from open_targets_release_26.09/
copied mondo.obo
NOT in Drive: who_eml.csv
copied B09474-eng.pdf
copied drugbank_smiles_cache.json
copied drugbank_cid_map.json


## Step 4 - Prepare: download PrimeKG + MONDO, build tables, save split, run the safety check, build the structure-covered variant
Stops at the first failure. A validation `[FAIL]` means: do not train.

In [ ]:
!python scripts/run_full_pipeline.py --stage prepare

Streaming output truncated to the last 5000 lines.
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:58:13] DEPRECATION WARNING: please use MorganGenerator
[22:5

### Step 4b - Verify the benchmark is the frozen one (do not continue on a mismatch)
Compares SHA-256 of the regenerated tables, features and saved splits with `Docs/release/vdm_v1_candidate_manifest.json` (frozen locally). If a processed file differs, the frozen copy from
`MyDrive/VDM_Data/frozen_processed/` is restored and re-checked. A differing `mondo.obo` or `primekg_kg.csv` is fatal (fix the input, do not train).

In [ ]:
import hashlib, json, os, shutil
man = json.load(open('Docs/release/vdm_v1_candidate_manifest.json'))['files']
KEEP = ('.csv', '.npy', '.npz', '.obo')
want = {f['path']: f['sha256'] for f in man if f['path'].endswith(KEEP) and
        f['path'].startswith(('data/processed/primekg/', 'data/processed/primekg_structcov/', 'data/external/mondo.obo', 'data/external/primekg_kg.csv'))}
def sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(1 << 22), b''):
            h.update(b)
    return h.hexdigest()
def check():
    return [p for p, h in want.items() if not os.path.exists(p) or sha(p) != h]
bad = check()
if bad:
    print('MISMATCH:', bad)
    fatal = [p for p in bad if p.startswith('data/external/')]
    assert not fatal, f'input differs from the frozen release: {fatal}'
    for name in ('primekg', 'primekg_structcov'):
        src = f'/content/drive/MyDrive/VDM_Data/frozen_processed/{name}'
        assert os.path.isdir(src), f'no frozen copy at {src}'
        shutil.copytree(src, f'data/processed/{name}', dirs_exist_ok=True)
    bad = check()
assert not bad, f'frozen benchmark NOT reproduced: {bad}'
print(f'FROZEN BENCHMARK VERIFIED: {len(want)} files match the release manifest')

## Step 5 - External evidence and the equity table (Open Targets, HAI, IHME, EML are optional inputs)

In [ ]:
!python scripts/run_full_pipeline.py --stage external

## Step 6 - Smoke test (do this before the long run)
Checks `torch_cluster`, Node2Vec for one epoch, five epochs for the five GNNs, one baseline. Look at runtime and GPU memory.

In [ ]:
%env MLFLOW_TRACKING_URI=sqlite:///mlflow.db
!python scripts/run_full_pipeline.py --stage smoke

## Step 6b - Make `outputs/` live on Google Drive and record the environment (do this once per session, after the smoke test)
The smoke test used 5-epoch models: its files are moved to a `smoke_and_setup_logs` folder and **removed from `outputs/`**, so they can never be mistaken for results.
From here on `outputs/` is a link to `MyDrive/VDM_Outputs/outputs`: every checkpoint, prediction, metric and log is written straight to Drive, and a re-run with `--resume` skips the models that already finished.

In [ ]:
import json, os, platform, shutil, subprocess, sys, time, hashlib
O = '/content/drive/MyDrive/VDM_Outputs'
stamp = time.strftime('%Y%m%d_%H%M%S')
os.makedirs(f'{O}/smoke_and_setup_logs/{stamp}', exist_ok=True)
if os.path.isdir('outputs') and not os.path.islink('outputs'):
    shutil.copytree('outputs', f'{O}/smoke_and_setup_logs/{stamp}/outputs', dirs_exist_ok=True)
    shutil.rmtree('outputs')
os.makedirs(f'{O}/outputs', exist_ok=True)
if not os.path.islink('outputs'):
    os.symlink(f'{O}/outputs', 'outputs')
import torch
def sh(c):
    return subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
cfgs = sorted(f for f in os.listdir('configs') if f.endswith('.yaml'))
env = {'recorded_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'git_commit': sh('git rev-parse HEAD'), 'git_describe_dirty': sh('git status --porcelain') != '',
       'python': sys.version, 'platform': platform.platform(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, 'seeds': 'split seed 42; baselines random_state 42; every GNN reseeded to 42 before training',
       'config_sha256': {f: hashlib.sha256(open(f'configs/{f}', 'rb').read()).hexdigest() for f in cfgs},
       'pip_freeze_lines': len(sh('pip freeze').splitlines())}
os.makedirs('outputs', exist_ok=True)
json.dump(env, open(f'outputs/run_environment_{stamp}.json', 'w'), indent=1)
open(f'outputs/pip_freeze_{stamp}.txt', 'w').write(sh('pip freeze'))
print(json.dumps(env, indent=1)[:900])
print('outputs ->', os.path.realpath('outputs'))

## Step 7 - Full benchmark: baselines, Node2Vec, five GNNs, cold-start ablation, then evaluation and the paper results package
Every model is one step. If Colab disconnects: run Steps 1-6b again (not 7 first), then re-run this cell: `--resume` skips the models that already finished at this commit (recorded in `outputs/pipeline_log.jsonl` on Drive) and trains only the rest.

In [ ]:
!python scripts/run_full_pipeline.py --stage train --variant full --resume
!python scripts/run_full_pipeline.py --stage report --variant full

## Step 7b - Explanations on TEST pairs (SHAP for Random Forest / XGBoost, GNNExplainer for HGT and RGCN)
Only after Step 7. Explains the saved cold-start models, not the pooled all-data model. Attributions describe the model, not causal biology.

In [ ]:
!python scripts/run_full_pipeline.py --stage explain --variant full

## Step 8 - Structure-covered variant (same drug partition, drugs with a usable fingerprint only)

In [ ]:
!python scripts/run_full_pipeline.py --stage train --variant structcov --resume
!python scripts/run_full_pipeline.py --stage report --variant structcov

## Step 9 - Freeze, package and download
Writes the release manifest (checksums, versions, commit), zips the results package, saves a copy to Drive and downloads it.

In [ ]:
!python scripts/freeze_release.py --tag vdm_v1
!python scripts/make_readiness_report.py --skip-validators
!zip -rq vdm_results.zip outputs/primekg/paper_package outputs/primekg_structcov/paper_package outputs/primekg/*.csv outputs/primekg_structcov/*.csv outputs/primekg_dp/*.csv outputs/primekg_structcov_dp/*.csv outputs/primekg/models/*.json outputs/primekg_structcov/models/*.json outputs/pipeline_log.jsonl outputs/pip_freeze_*.txt outputs/run_environment_*.json outputs/primekg/explain Docs/release data/processed/equity data/processed/ntd data/processed/open_targets data/processed/hai
!cp vdm_results.zip /content/drive/MyDrive/VDM_Outputs/
from google.colab import files
files.download('vdm_results.zip')

## Reading the results honestly
Report the `all` rows first; show `with_structure` / `no_structure` next to them; NTD rows have ~15 test positives (exploratory). Baselines come in two variants (`primekg` = drug-only; `primekg_dp` = drug + train-side disease profile): report both, never pooled. Use only `coldstart_*` baseline numbers (the pooled 5-fold CV is not run: baselines use `--coldstart-only`). The legacy `train_benchmark.py` Node2Vec and `run_ablations.py` are not drug-disjoint: never report them.